# 2-stage detection — unified backbone-matrix training

Trains the **10-backbone detection comparison** (CNN vs ViT, supervised vs SSL, plus
native foundation models: DINOv2, DINOv2+reg, AM-RADIO) across all 6 subsets —
DIVA-HisDB (CB55, CS18, CS863) + U-DIADS-TL (Latin14396, Latin2, Syr341).

Backbone selection lives in `train_detector.py`'s `BACKBONES` registry and mirrors
`01_simple_segmentation` exactly. Transformer backbones use hier_encoder's SFP neck;
CNNs use stock YOLOv8 detection YAMLs. Select a GPU runtime before starting. Results
are copied to Drive after every model, so rerunning resumes safely.

## Setup (Colab): mount Drive, unpack repo, install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
ARCHIVE = Path('/content/drive/MyDrive/two_stage_colab.zip')
RESULTS = Path('/content/drive/MyDrive/two_stage_detection_results')
assert ARCHIVE.exists(), f'Upload the repo bundle to {ARCHIVE}'
RESULTS.mkdir(parents=True, exist_ok=True)

In [ ]:
!rm -rf /content/fsl-tl-manuscripts
!unzip -q "{ARCHIVE}" -d /content
%cd /content/fsl-tl-manuscripts
!pip -q install -e '.[all]' ultralytics opencv-python-headless Pillow PyYAML
!nvidia-smi

DINOv3 / DINOv2 weights may require accepting Meta's license and authenticating with
Hugging Face. If access is gated, run the login cell. The SFP loader refuses a
random-weight fallback (it raises instead of silently training on noise).

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()

## Quick check: the 10-backbone matrix is registered

In [ ]:
%cd /content/fsl-tl-manuscripts/50_modelling/02_2stage
import sys; sys.path.insert(0, '.')
import train_detector
for i, (name, (kind, ref)) in enumerate(train_detector.BACKBONES.items(), 1):
    print(f'{i}. {name:38} [{kind:4}] -> {ref}')

## Train the full matrix

Drives `run_train_all.sh` (10 backbones x 6 subsets = 60 checkpoints). Skips any run
whose `weights/best.pt` already exists, so the notebook resumes safely. Tune the loop
with env vars: `IMGSZ`, `BATCH`, `DIVA_EPOCHS`, `UDIADS_EPOCHS`.

In [ ]:
import os
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ['IMGSZ'] = '1280'
os.environ['BATCH'] = '2'
os.environ['DIVA_EPOCHS'] = '100'
os.environ['UDIADS_EPOCHS'] = '100'
!./run_train_all.sh

### Single-model run (debug / one backbone-subset)

In [ ]:
# !python train_detector.py --encoder dinov2 \
#     --data configs/data/diva_cb55_detect.yaml \
#     --project /content/out --name yolov8_dinov2_diva_CB55 \
#     --epochs 100 --imgsz 1280 --batch 2

## Sync checkpoints to Drive + progress summary

In [ ]:
import shutil
src = Path('../../80_models/02_2stage')
completed = sorted(src.rglob('weights/best.pt'))
print(f'{len(completed)}/60 checkpoints present')
for ckpt in completed:
    rel = ckpt.relative_to(src)
    dst = RESULTS / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(ckpt, dst)
    print(' ', rel.parent.parent)